# 序号26：回测陷阱专题

## 学习目标
- 亲手制造至少 5 种"回测好看、实盘拉胯"的典型案例
- 学会识别：幸存者偏差、前视偏差、过拟合、数据窥探、忽略交易成本
- 养成质疑回测结果的习惯：每看到一个漂亮曲线，先问"哪里可能错了？"

## 验收标准
- ✅ 能识别至少 5 种回测陷阱
- ✅ 知道如何避免每种陷阱
- ✅ 养成质疑回测结果的习惯

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import akshare as ak
import warnings, time
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti SC', 'STHeiti', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

print('✅ 环境就绪')

## 0. 数据准备

选取一只代表性的沪深300成分股（贵州茅台 600519）作为演示标的，时间跨度 5 年。

In [ ]:
# 获取数据
print('📡 获取贵州茅台日线数据...')
df_raw = ak.stock_zh_a_hist(symbol='600519', period='daily',
                            start_date='20210101', end_date='20260620',
                            adjust='qfq')
df_raw['日期'] = pd.to_datetime(df_raw['日期'])
df = df_raw.set_index('日期').sort_index()
df['ret'] = df['收盘'].pct_change()
df = df.dropna()

print(f'数据范围: {df.index[0].date()} ~ {df.index[-1].date()}')
print(f'交易日数: {len(df)}')
print(f'累计收益: {df["ret"].add(1).prod() - 1:.2%}')
df[['收盘', 'ret']].tail(3)

---

## 🔴 陷阱一：幸存者偏差（Survivorship Bias）

### 什么是幸存者偏差？

用**现在还在的股票**回测历史策略，忽略了已经退市/ST/被并购的股票。那些"死去"的股票往往是表现最差的，剔除它们会让回测结果系统性偏高。

### 案例演示

我们用沪深300 **当前**成分股回测 vs **历史各期**成分股回测，对比差异。

In [ ]:
# 获取当前沪深300成分股（幸存者）
print('📡 获取当前沪深300成分股...')
hs300_now = ak.index_stock_cons_csindex(symbol='000300')
survivors = hs300_now['成分券代码'].tolist()[:30]  # 取前30只
print(f'当前成分股(前30): {survivors[:10]}...')

# 用这30只"幸存者"做历史回测
print('\n📡 获取30只股票近5年数据...')
price_survivor = {}
for i, code in enumerate(survivors):
    try:
        d = ak.stock_zh_a_hist(symbol=code, period='daily',
                               start_date='20210101', end_date='20260620',
                               adjust='qfq')
        d['日期'] = pd.to_datetime(d['日期'])
        d = d.set_index('日期').sort_index()
        if len(d) > 500:
            price_survivor[code] = d['收盘']
    except: pass
    if (i+1) % 10 == 0:
        print(f'  进度: {i+1}/{len(survivors)}')
    time.sleep(0.15)

price_s_df = pd.DataFrame(price_survivor)
print(f'\n成功获取: {len(price_survivor)}只')

# 等权组合收益
eq_ret_survivor = price_s_df.pct_change().mean(axis=1)
cum_survivor = (1 + eq_ret_survivor).cumprod()
print(f'"幸存者"等权组合累计收益: {cum_survivor.iloc[-1] - 1:.2%}')
print(f'年化收益: {(cum_survivor.iloc[-1]) ** (252/len(cum_survivor)) - 1:.2%}')

In [ ]:
# 获取历史各期成分股（取中间某期做对比）
# 使用2022年6月的沪深300成分股
print('📡 获取2022年6月的沪深300成分股（历史成分股）...')
# akshare 不支持历史成分股查询，我们用折中方案：
# 模拟"非幸存者"效应：在当前成分股中加入已退市的乐视网（300104，2020年退市）

# 加入一只"已死"股票的数据（如果能获取到退市前的）
# 这里用一只表现很差的股票来模拟退市股效应
print('⚠️ 由于API限制，用模拟方式演示幸存者偏差：')
print('假设有20%的股票在回测期间退市，平均亏损-60%...')

# 模拟：实际可投资股票池包含20%后来退市的股票
n_survivors = len(cum_survivor)
# 构造"含退市股"的收益序列：幸存者收益 * 0.8 + 退市股收益 * 0.2
# 退市股平均亏损60%，分5年均匀实现
np.random.seed(42)
bust_ret = np.random.normal(-0.003, 0.025, n_survivors)  # 退市股日收益（负期望）
bust_ret = pd.Series(bust_ret, index=cum_survivor.index)

# 含退市股的真实组合收益
eq_ret_true = eq_ret_survivor * 0.8 + bust_ret * 0.2
cum_true = (1 + eq_ret_true).cumprod()

print(f'\n"幸存者"组合累计收益: {cum_survivor.iloc[-1] - 1:.2%}')
print(f'"含退市股"真实组合累计收益: {cum_true.iloc[-1] - 1:.2%}')
print(f'偏差: {(cum_survivor.iloc[-1] - cum_true.iloc[-1]):.2%}')

In [ ]:
# 可视化
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cum_survivor.index, cum_survivor.values, label='只用幸存者回测 (偏差)', 
        color='#dc2626', linewidth=2)
ax.plot(cum_true.index, cum_true.values, label='含退市股的真实组合', 
        color='#2563eb', linewidth=2)
ax.fill_between(cum_survivor.index, cum_survivor.values, cum_true.values, 
                 alpha=0.15, color='red', label='幸存者偏差')
ax.set_title('陷阱一：幸存者偏差', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print('💡 教训：永远用当时可投资的股票池回测，而不是现在的成分股')

---

## 🔴 陷阱二：前视偏差（Look-ahead Bias）

### 什么是前视偏差？

在 t 时刻做决策时，**不小心用了 t 时刻之后才存在的信息**。最常见的三种：

1. **用全样本统计量做信号**：比如用整个回测期的均值和标准差来计算 Z-score
2. **财务数据发布时间错位**：t 日用到了 t 日才公布的季报数据
3. **用调整后的数据回测**：比如用复权价回测但假设你能在复权价上交易

### 案例演示

场景：构建一个布林带策略，"不小心"用整个回测期的均值和标准差来计算上下轨。

In [ ]:
# === 前视偏差版：用全样本统计量计算布林带 ===
# （这是错误做法——你在回测第一天就知道了整个历史的价格范围）

price = df['收盘']

# 全样本均值和标准差（前视！）
mean_all = price.mean()
std_all = price.std()

upper_biased = mean_all + 2 * std_all
lower_biased = mean_all - 2 * std_all

# 错误信号：低于全样本下轨就买，高于上轨就卖
signal_biased = pd.Series(0, index=df.index)
signal_biased[price < lower_biased] = 1   # 买入
signal_biased[price > upper_biased] = -1  # 卖出
signal_biased = signal_biased.shift(1).fillna(0)  # t日信号用于t+1日

# 策略收益
strat_ret_biased = signal_biased * df['ret']
cum_biased = (1 + strat_ret_biased).cumprod()

print(f'前视偏差版布林带策略:')
print(f'  累计收益: {cum_biased.iloc[-1] - 1:.2%}')
print(f'  年化夏普: {strat_ret_biased.mean() / strat_ret_biased.std() * np.sqrt(252):.2f}')

In [ ]:
# === 正确版：用滚动窗口（expanding window）计算布林带 ===
# 每天的均值和std只用该日之前的数据

window = 252  # 至少1年数据才开始交易
roll_mean = price.rolling(window).mean()
roll_std = price.rolling(window).std()

upper_correct = roll_mean + 2 * roll_std
lower_correct = roll_mean - 2 * roll_std

signal_correct = pd.Series(0, index=df.index)
signal_correct[price < lower_correct] = 1
signal_correct[price > upper_correct] = -1
signal_correct = signal_correct.shift(1).fillna(0)

strat_ret_correct = signal_correct * df['ret']
cum_correct = (1 + strat_ret_correct).cumprod()

print(f'正确版滚动布林带策略:')
print(f'  累计收益: {cum_correct.iloc[-1] - 1:.2%}')
print(f'  年化夏普: {strat_ret_correct[window:].mean() / strat_ret_correct[window:].std() * np.sqrt(252):.2f}')
print(f'\n前视偏差导致的收益虚增: {(cum_biased.iloc[-1] - cum_correct.iloc[-1]):.2%}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# 上图：布林带对比
ax = axes[0]
ax.plot(price.index, price.values, color='gray', alpha=0.7, linewidth=0.8, label='价格')
ax.axhline(y=upper_biased, color='red', linestyle='--', alpha=0.5, label=f'全样本上轨={upper_biased:.0f}')
ax.axhline(y=lower_biased, color='red', linestyle='--', alpha=0.5, label=f'全样本下轨={lower_biased:.0f}')
ax.plot(price.index[-500:], roll_mean.iloc[-500:], color='blue', linewidth=1.5, alpha=0.7, label='滚动均值')
ax.plot(price.index[-500:], upper_correct.iloc[-500:], color='green', linewidth=1, alpha=0.5, label='滚动布林带')
ax.plot(price.index[-500:], lower_correct.iloc[-500:], color='green', linewidth=1, alpha=0.5)
ax.set_title('前视偏差演示：全样本布林带 vs 滚动布林带', fontsize=13, fontweight='bold')
ax.legend(fontsize=8, loc='upper left')
ax.grid(True, alpha=0.3)

# 下图：收益对比
ax = axes[1]
ax.plot(cum_biased.index, cum_biased.values, color='red', linewidth=2, label='前视偏差版（好看但假）')
ax.plot(cum_correct.index, cum_correct.values, color='blue', linewidth=2, label='滚动版（真实）')
ax.set_title('策略收益对比', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print('💡 教训：任何信号计算都只能用该时点之前的数据。expanding/rolling window 是你的朋友。')

---

## 🔴 陷阱三：过拟合（Overfitting）

### 什么是过拟合？

参数优化时，策略不仅学到了真实的规律，还学到了**样本内的噪音**。样本内表现越好，样本外可能越差。

### 经典症状
- 参数越多，过拟合风险越大
- 样本内夏普 > 3，样本外大概率不可持续
- 参数微调导致收益剧烈变化（不稳定）

In [ ]:
# 演示：过度参数优化
# 对一个简单的均线策略做网格搜索

# 划分样本内外
split_date = '2024-06-01'
train = df[df.index < split_date]
test = df[df.index >= split_date]

print(f'样本内: {train.index[0].date()} ~ {train.index[-1].date()}, {len(train)}天')
print(f'样本外: {test.index[0].date()} ~ {test.index[-1].date()}, {len(test)}天')

# 网格搜索
results = []
for short_ma in range(2, 61, 2):
    for long_ma in range(short_ma+5, 121, 2):
        if long_ma <= short_ma: continue
        
        ma_short = train['收盘'].rolling(short_ma).mean()
        ma_long = train['收盘'].rolling(long_ma).mean()
        
        signal = pd.Series(0, index=train.index)
        signal[ma_short > ma_long] = 1
        signal[ma_short < ma_long] = -1
        signal = signal.shift(1).fillna(0)
        
        ret = signal[long_ma:] * train['ret'][long_ma:]
        sharpe = ret.mean() / ret.std() * np.sqrt(252) if ret.std() > 0 else 0
        total_ret = (1 + ret).prod() - 1
        
        results.append({
            'short': short_ma, 'long': long_ma,
            'sharpe': sharpe, 'total_ret': total_ret,
            'n_params': 2  # 简单策略，参数少
        })

results_df = pd.DataFrame(results)
print(f'共测试 {len(results_df)} 组参数')

# 最佳参数（样本内）
best = results_df.loc[results_df['sharpe'].idxmax()]
print(f'\n样本内最佳: MA({int(best["short"])}, {int(best["long"])})')
print(f'  样本内夏普: {best["sharpe"]:.2f}')
print(f'  样本内收益: {best["total_ret"]:.2%}')

In [ ]:
# 用最佳参数在样本外测试
short_best, long_best = int(best['short']), int(best['long'])

ma_s_oos = test['收盘'].rolling(short_best).mean()
ma_l_oos = test['收盘'].rolling(long_best).mean()

signal_oos = pd.Series(0, index=test.index)
signal_oos[ma_s_oos > ma_l_oos] = 1
signal_oos[ma_s_oos < ma_l_oos] = -1
signal_oos = signal_oos.shift(1).fillna(0)

ret_oos = signal_oos[long_best:] * test['ret'][long_best:]
sharpe_oos = ret_oos.mean() / ret_oos.std() * np.sqrt(252) if ret_oos.std() > 0 else 0
total_oos = (1 + ret_oos).prod() - 1

print(f'样本外表现 (MA({short_best}, {long_best})):')
print(f'  样本外夏普: {sharpe_oos:.2f}')
print(f'  样本外收益: {total_oos:.2%}')
print(f'\n⚠️ 夏普衰减: {best["sharpe"]:.2f} → {sharpe_oos:.2f} ({(best["sharpe"] - sharpe_oos):.2f})')
print(f'⚠️ 收益衰减: {best["total_ret"]:.2%} → {total_oos:.2%}')

In [ ]:
# 可视化：参数稳定性——最优参数附近的夏普分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左：样本内夏普热力图
pivot = results_df.pivot_table(values='sharpe', index='short', columns='long')
ax = axes[0]
im = ax.contourf(pivot.columns, pivot.index, pivot.values, levels=20, cmap='RdYlGn')
ax.scatter([long_best], [short_best], c='blue', s=100, marker='*', 
           edgecolors='white', linewidth=1, zorder=5, label=f'最优 ({short_best},{long_best})')
ax.set_xlabel('Long MA', fontsize=11)
ax.set_ylabel('Short MA', fontsize=11)
ax.set_title('样本内夏普热力图', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
plt.colorbar(im, ax=ax, label='Sharpe')

# 右：样本内 vs 样本外夏普散点图
ax = axes[1]
# 取部分参数做散点
sample_n = min(500, len(results_df))
sample_results = results_df.sample(sample_n, random_state=42)

# 计算这些参数在样本外的夏普
oos_sharpes = []
for _, row in sample_results.iterrows():
    s, l = int(row['short']), int(row['long'])
    ms = test['收盘'].rolling(s).mean()
    ml = test['收盘'].rolling(l).mean()
    sig = pd.Series(0, index=test.index)
    sig[ms > ml] = 1; sig[ms < ml] = -1
    sig = sig.shift(1).fillna(0)
    r = sig[l:] * test['ret'][l:]
    sr = r.mean() / r.std() * np.sqrt(252) if r.std() > 0 else 0
    oos_sharpes.append(sr)

sample_results['sharpe_oos'] = oos_sharpes

ax.scatter(sample_results['sharpe'], sample_results['sharpe_oos'], 
           alpha=0.5, s=20, c='steelblue')
ax.scatter([best['sharpe']], [sharpe_oos], c='red', s=150, marker='*', 
           edgecolors='white', linewidth=1, zorder=5, label='样本内最佳参数')
lims = [min(sample_results['sharpe'].min(), sample_results['sharpe_oos'].min()) - 0.1,
        max(sample_results['sharpe'].max(), sample_results['sharpe_oos'].max()) + 0.1]
ax.plot(lims, lims, '--', color='gray', alpha=0.5, label='y=x (无衰减)')
ax.set_xlabel('样本内夏普', fontsize=11)
ax.set_ylabel('样本外夏普', fontsize=11)
ax.set_title('过拟合诊断：样本内 vs 样本外', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print('💡 如果散点在 y=x 线以下大面积分布 → 过拟合严重')
print('💡 样本内最佳参数往往是"运气+技能"的混合物')

---

## 🔴 陷阱四：忽略交易成本（Ignoring Transaction Costs）

**"回测里赚的钱，扣完手续费可能就没了。"**

常见的被忽略的成本：
- 手续费（佣金 + 印花税）：A 股约 0.06%-0.15%/次
- 滑点：大单冲击市场价格
- 买卖价差：流动性差的股票尤其严重
- 冲击成本：资金量越大越明显

In [ ]:
# 用双均线策略演示：加成本 vs 不加成本
short_ma, long_ma = 5, 20  # 较频繁交易

price_full = df['收盘']
ma_s = price_full.rolling(short_ma).mean()
ma_l = price_full.rolling(long_ma).mean()

signal = pd.Series(0, index=df.index)
signal[ma_s > ma_l] = 1
signal[ma_s < ma_l] = -1
signal = signal.shift(1).fillna(0)

# 无成本版
ret_no_cost = signal[long_ma:] * df['ret'][long_ma:]
cum_no_cost = (1 + ret_no_cost).cumprod()

# 有成本版（双边0.1%）
cost_rate = 0.001  # 0.1%
turnover = signal.diff().abs()  # 换手信号（每次变化 = 一次买卖）
cost = turnover * cost_rate
ret_with_cost = ret_no_cost - cost[long_ma:]
cum_with_cost = (1 + ret_with_cost).cumprod()

print(f'无成本版:')
print(f'  累计收益: {cum_no_cost.iloc[-1] - 1:.2%}')
print(f'  年化夏普: {ret_no_cost.mean() / ret_no_cost.std() * np.sqrt(252):.2f}')
print(f'  交易次数: {int(turnover.sum())}')

print(f'\n有成本版 (双边0.1%):')
print(f'  累计收益: {cum_with_cost.iloc[-1] - 1:.2%}')
print(f'  年化夏普: {ret_with_cost.mean() / ret_with_cost.std() * np.sqrt(252):.2f}')
print(f'  总成本: {cost.sum():.2%}')
print(f'\n⚠️ 成本"吃掉"了 {(cum_no_cost.iloc[-1] - cum_with_cost.iloc[-1]):.2%} 的收益！')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(cum_no_cost.index, cum_no_cost.values, color='red', linewidth=2, label='无成本（虚假）')
ax.plot(cum_with_cost.index, cum_with_cost.values, color='blue', linewidth=2, label='含0.1%成本（真实）')
ax.set_title('交易成本对收益的影响', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.grid(True, alpha=0.3)

# 不同成本率下的收益衰减
ax = axes[1]
cost_rates = np.linspace(0, 0.005, 20)
final_rets = []
for cr in cost_rates:
    c = turnover * cr
    r = ret_no_cost - c[long_ma:]
    final_rets.append((1 + r).prod() - 1)

ax.plot(cost_rates * 100, [r * 100 for r in final_rets], 
        color='steelblue', linewidth=2)
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.5)
ax.axvline(x=0.1, color='red', linestyle='--', alpha=0.5, label='A股 ~0.1%')
ax.set_xlabel('单边成本 (%)', fontsize=11)
ax.set_ylabel('最终收益 (%)', fontsize=11)
ax.set_title('不同成本率下的最终收益', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print('💡 高频策略对成本极度敏感，成本率提高0.1%可能直接亏损')

---

## 🔴 陷阱五：数据窥探（Data Snooping）

### 什么是数据窥探？

你试了 100 个策略，挑了最好的那个展示——但实际上那个好结果只是随机中的幸运儿。

**数学本质**：
- 假设每个策略的真实夏普 = 0（无效策略）
- 试 100 个策略，最大的夏普期望值 ≈ σ × √(2 log n) ≈ 3σ
- 即试 100 次，纯靠运气也能"发现"夏普 > 2 的策略

### 案例：随机信号实验

In [ ]:
# 随机信号实验：证明即使完全随机的策略，试多了也会出现"好"结果
np.random.seed(123)

n_trials = 200  # 试200个随机策略
n_days = len(df['ret'][long_ma:])

random_sharpes = []
random_rets = []

for i in range(n_trials):
    # 随机生成 +1/-1 信号
    rand_signal = np.random.choice([-1, 1], size=n_days)
    rand_ret = rand_signal * df['ret'][long_ma:].values
    sr = rand_ret.mean() / rand_ret.std() * np.sqrt(252) if rand_ret.std() > 0 else 0
    random_sharpes.append(sr)
    random_rets.append((1 + rand_ret).prod() - 1)

random_sharpes = np.array(random_sharpes)
random_rets = np.array(random_rets)

print(f'===== 随机信号实验 ({n_trials} 次) =====')
print(f'平均夏普: {random_sharpes.mean():.2f}')
print(f'夏普标准差: {random_sharpes.std():.2f}')
print(f'最佳夏普: {random_sharpes.max():.2f} ← 纯靠运气！')
print(f'夏普>1.0 的比例: {(random_sharpes > 1.0).mean():.1%}')
print(f'夏普>2.0 的比例: {(random_sharpes > 2.0).mean():.1%}')
print(f'\n理论最大值 ≈ σ × √(2×log({n_trials})) = {random_sharpes.std():.2f} × √(2×log({n_trials})) ≈ {random_sharpes.std() * np.sqrt(2*np.log(n_trials)):.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(random_sharpes, bins=40, color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(x=random_sharpes.max(), color='red', linestyle='--', linewidth=2, 
           label=f'最佳={random_sharpes.max():.2f} (纯运气)')
ax.axvline(x=0, color='gray', linestyle='-')
ax.set_xlabel('年化夏普', fontsize=11)
ax.set_ylabel('频次', fontsize=11)
ax.set_title(f'随机策略夏普分布 (n={n_trials})', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

ax = axes[1]
# 最佳随机策略 vs 买入持有
best_idx = random_sharpes.argmax()
best_rand_signal = np.random.RandomState(123 + best_idx).choice([-1, 1], size=n_days)
best_rand_ret = pd.Series(best_rand_signal * df['ret'][long_ma:].values, index=df.index[long_ma:])
cum_best_rand = (1 + best_rand_ret).cumprod()
cum_bh = (1 + df['ret'][long_ma:]).cumprod()

ax.plot(cum_best_rand.index, cum_best_rand.values, color='red', linewidth=2, 
        label=f'最佳随机策略 (夏普={random_sharpes.max():.2f})')
ax.plot(cum_bh.index, cum_bh.values, color='gray', linewidth=1.5, alpha=0.7, label='买入持有')
ax.set_title('看起来"最好"的随机策略', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print('💡 试了200个策略后，总会有一个"夏普>2"的。这不代表策略有效——只是你试得够多。')
print('💡 应对：Bonferroni校正、样本外验证、经济逻辑先行')

---

## 六大回测陷阱总结

| # | 陷阱 | 表现 | 应对方法 |
|---|------|------|----------|
| 1 | **幸存者偏差** | 只用现存股票回测 | 用历史成分股 / Point-in-Time 数据 |
| 2 | **前视偏差** | 信号用了未来信息 | 严格用 expanding/rolling window |
| 3 | **过拟合** | 样本内好、样本外差 | 样本外保留、简单策略优先、参数少 |
| 4 | **忽略交易成本** | 回测收益被成本吞掉 | 加入手续费+滑点+冲击成本 |
| 5 | **数据窥探** | 试了N个挑最好的 | Bonferroni校正、经济逻辑驱动 |
| 6 | **样本选择偏差** | 只在一个标的上好使 | 多标的、多时段、多市场验证 |

## 黄金法则

> **任何回测在样本外验证和实盘之前，都是"好看的故事"——不是证据。**

1. 样本外数据是圣杯——回测阶段绝对不能碰
2. 策略越简单越好——参数越多，过拟合风险越大
3. 成本是真实的——不加成本的夏普不算数
4. 问自己：这个策略有经济逻辑吗？还是纯数据挖掘？
5. 如果夏普 > 2，先怀疑自己哪里写错了

## 验收自检

- [x] 能识别幸存者偏差：只看现存股票回测
- [x] 能识别前视偏差：信号用了未来数据
- [x] 能识别过拟合：样本内好样本外差
- [x] 能识别交易成本影响：扣费前后差异巨大
- [x] 能识别数据窥探：试多了总有好结果
- [x] 养成质疑习惯：看到漂亮曲线先想"哪里错了"

> **下一步**：序号27 — 绩效归因分析，学会拆解收益来源，区分运气和能力。